# **LangChain and LangGraph**

## **01. The four levels of abstraction**

LangChain and LangGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because 'LangChain' appears in a few places..

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |


### **Layer 1: the building blocks**

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

#### **01. A first model call**

`invoke` with a prompt; this is a key method in LangChain.

In [6]:
llm = ChatOpenAI(model="gpt-5.4-mini")

message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply = llm.invoke(message)

print(reply.content)

An AI agent is autonomous if it can independently perceive its environment, make decisions, and take actions to achieve goals with minimal human intervention.


#### **02. Streaming**

For a live, token by token feel, `stream` and loop over the chunks

In [7]:
for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Silent minds in motion, seeking paths through shifting code,  
Autonomous agents dream in loops, then choose where worlds unfold.

#### **03. Messages**

LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [9]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of Deutschland?"),
]

print(llm.invoke(messages).content)

# The exact same call using plain dictionaries
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of Deutschland?"},
]
print(llm.invoke(messages_as_dicts).content)

Berlin is the capital city.
Berlin is Germany's capital city.


#### **04. Tools with the `@tool` decorator**

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.


In [10]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


In [18]:
@tool
def percentage_change(old_value: float, new_value: float) -> float:
    """Calculate the percentage change from an old value to a new value."""
    if old_value == 0:
        return 0.0
    return ((new_value - old_value) / old_value) * 100

print("name:", percentage_change.name)
print("description:", percentage_change.description)
print("args:", percentage_change.args)
print("called directly:", percentage_change.invoke({"old_value": 120, "new_value": 150}))

name: percentage_change
description: Calculate the percentage change from an old value to a new value.
args: {'old_value': {'title': 'Old Value', 'type': 'number'}, 'new_value': {'title': 'New Value', 'type': 'number'}}
called directly: 25.0


#### **05. Giving tools to the model**

we have to write the loop ourselves.

The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [19]:
llm_with_tools = llm.bind_tools([get_share_price, percentage_change])

response = llm_with_tools.invoke(
    "What is Amazon's share price, and what is the percentage change from 120 to 150? and what is Neura Robotics"
)

print("content:", response.content)
print("content repr:", repr(response.content))
print("tool_calls:", response.tool_calls)
print("content:", response.content)
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: 
content repr: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_3l5aOVLKNVAoQoifsTT0L38M', 'type': 'tool_call'}]
content: 
content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_3l5aOVLKNVAoQoifsTT0L38M', 'type': 'tool_call'}]


#### **06. Running the tool loop by hand**

In [22]:
conversation = [
    HumanMessage(
        "What is Amazon's share price, and what is the percentage change from 120 to 150?"
    )
]

ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(
            ToolMessage(content=str(result), tool_call_id=call["id"])
        )

    elif call["name"] == "percentage_change":
        result = percentage_change.invoke(call["args"])
        conversation.append(
            ToolMessage(content=str(result), tool_call_id=call["id"])
        )

final = llm_with_tools.invoke(conversation)
print("FINAL CONTENT:", repr(final.content))
print("FINAL TOOL CALLS:", final.tool_calls)

FINAL CONTENT: ''
FINAL TOOL CALLS: [{'name': 'percentage_change', 'args': {'old_value': 120, 'new_value': 150}, 'id': 'call_lLKZOSpGoM4sU22gKnKiSYPR', 'type': 'tool_call'}]
